# Data Cleaning

This notebook implements the **data preparation stage** of the fraud detection pipeline.  

It **loads and inspects** the raw transaction data, performs **feature engineering and behavioural enrichment**, and constructs a **stratified 5M-row sample**.

The processed dataset and corresponding **train/test splits** are **saved to disk** for reuse in downstream **supervised and behavioural modeling**.


## 1.Setup environment and load the Dataset

In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#import required libaries

import pandas as pd
import numpy as np
import os
import joblib

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
#load the dataset from HuggingFace
ds = load_dataset('CiferAI/Cifer-Fraud-Detection-Dataset-AF')

In [ ]:
ds.shape

In [ ]:
#convert the dataset to pandas DataFrame
df=ds['train'].to_pandas()

## 2.Inspect Raw Data

In [ ]:
#check the count of fraud and non-fraud cases
df['isFraud'].value_counts()

In [ ]:
#check for missing values
df.isnull().sum()

In [ ]:
#check for duplicates
df.duplicated().sum()

In [ ]:
#check data types
df.dtypes

In [ ]:
#check if there are invlaid amount
(df['amount']<0).sum()

In [ ]:
#check for min number of steps
df['step'].min()

In [ ]:
#check for max number of steps
df['step'].max()

## 3.Construct Stratified Sample

In [ ]:
#construct a stratified 5M-row sample preserving all fraud cases

#set sample size
sample_size=5000000

#separate fraud and non-fraud classes
fraud_df=df[df['isFraud']== 1]
nonfraud_df=df[df['isFraud']== 0]

#sample non-fraud rows to fill the remaining rows
nonfraud_sample_size=sample_size-len(fraud_df)
nonfraud_sample= nonfraud_df.sample(n=nonfraud_sample_size, random_state=42)

#combine all fraud rows with sampled non-fraud rows
sample_df=pd.concat([nonfraud_sample, fraud_df]).reset_index(drop=True)

#shuffle rows randomly
sample_df=sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

#check counts and proportions
print(sample_df['isFraud'].value_counts())
print(sample_df['isFraud'].value_counts(normalize=True))

In [ ]:
sample_df.info()

## 4.Engineer Features

In [ ]:
#derive hour and day features

sample_df['day']=(sample_df['step']//24)+1
sample_df['hour']=sample_df['step']%24

In [ ]:
#create a new column capturing balance change for origin and destination accounts

sample_df['balance_change_orig']=sample_df['oldbalanceOrg']-sample_df['newbalanceOrig']
sample_df['balance_change_dest']=sample_df['oldbalanceDest']-sample_df['newbalanceDest']

In [ ]:
#create a new column flagging transactions in the top 5% amount within each transaction type

sample_df['type_high_amount']=sample_df.groupby('type')['amount'].transform(lambda x: x>x.quantile(0.95)).astype(int)

In [ ]:
#create a new column to categorise hour into part of the day
def part_of_day(hour):
    if 5 <= hour < 12:
      return 'morning'
    elif 12 <= hour < 17:
      return 'afternoon'
    elif 17 <=hour <21:
      return 'evening'
    else:
      return 'night'

sample_df['part_of_day']=sample_df['hour'].apply(part_of_day)

In [ ]:
#create a new column splitting transaction amounts into 4 quantile-based bins

sample_df['amount_quantile']=pd.qcut(sample_df['amount'], 4, labels=False)

In [ ]:
#create a new columns representing the ratio of transaction amount to the sender’s and receiver’s balance

sample_df['amount_to_oldbalance_ratio']=sample_df['amount']/(sample_df['oldbalanceOrg']+1)
sample_df['amount_to_destbalance_ratio']=sample_df['amount']/(sample_df['oldbalanceDest']+1)

In [ ]:
sample_df.info()

## 5.Save Sampled Dataset

In [ ]:
#saved the sampled dataset to Google Drive for later modeling

data_folder='/content/drive/MyDrive/hybrid-fraud-risk-prioritisation/data'
os.makedirs(data_folder, exist_ok=True)

sample_file_path=os.path.join(data_folder, 'sample_5M_df.parquet')
sample_df.to_parquet(sample_file_path, index=False)

print(f"Sample df saved sucessfully at: {sample_file_path}")

## 6.Prepare data for modeling

In [ ]:
#drop id columns as they are high-cardinality identifiers and not directly informative for modeling

sample_df=sample_df.drop(columns=['nameOrig', 'nameDest'])

In [ ]:
sample_df.head()

In [ ]:
#separate features and target

X=sample_df.drop('isFraud', axis=1)
y=sample_df['isFraud']

In [ ]:
#split the dataset into 80% training, 20% test

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
#encode categorical features using ordinal encoding
categorical_features=['type', 'part_of_day']

ord_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train[categorical_features] = ord_enc.fit_transform(X_train[categorical_features])
X_test[categorical_features] = ord_enc.transform(X_test[categorical_features])

## 7.Save Processed Train/Test Splits

In [ ]:
#save preprocessed train/test datasets for for later supervised modeling

joblib.dump(X_train, f'{data_folder}/X_train.pkl')
joblib.dump(y_train, f'{data_folder}/y_train.pkl')
joblib.dump(X_test, f'{data_folder}/X_test.pkl')
joblib.dump(y_test, f'{data_folder}/y_test.pkl')

print('Processed train and test datasets saved successfully.')